# 📚 Lesson 3 — Modifying DataFrames & GroupBy

**Time:** ~2.5 hours
**Goal:** Learn to add/remove columns, rename, change types, and group data.

---

## 🧠 Concept: Changing Your Data

Once you have a DataFrame, you'll want to:
1. **Add new columns** (calculate things, combine data)
2. **Remove columns** (drop what you don't need)
3. **Rename columns** (make names clearer)
4. **Change data types** (fix wrong types)
5. **Group and summarize** (the most powerful pandas feature!)

---

## 📖 Part 1: Adding Columns

### Method A: Direct Assignment

```python
df['NewColumn'] = [1, 2, 3]           # from a list
df['NewColumn'] = df['A'] + df['B']   # from existing columns
df['NewColumn'] = 0                    # same value for all rows
```

### Method B: `.assign()` — Returns a NEW DataFrame

```python
df_new = df.assign(
    NewCol1=df['A'] * 2,
    NewCol2=lambda x: x['NewCol1'] + 10  # can reference other new columns!
)
```

**Key difference:** `.assign()` does NOT modify the original — it returns a new one. Use it when you want to chain operations.

---

## 📖 Part 2: Removing Columns

```python
df.drop('ColumnName', axis=1)          # drop one column
df.drop(['Col1', 'Col2'], axis=1)     # drop multiple columns
```

**Note:** `drop()` returns a new DataFrame by default. To modify in place:
```python
df.drop('ColumnName', axis=1, inplace=True)
```

---

## 📖 Part 3: Renaming and Changing Index

```python
# Rename columns
df.rename(columns={'old_name': 'new_name'})

# Set a column as the index
df.set_index('ColumnName')

# Reset index back to default (0, 1, 2, ...)
df.reset_index()
```

---

## 📖 Part 4: Changing Data Types

Sometimes pandas guesses wrong. You might load a number column as text!

```python
# Change one column's type
df['Price'] = df['Price'].astype(float)

# Change multiple columns at once
df = df.astype({'Age': int, 'Name': str, 'Score': float})
```

---

## 📖 Part 5: GroupBy — The Power Tool

**GroupBy = Split → Apply → Combine**

Think of it like a pivot table in Excel:

```python
# Split by Category, then apply aggregations
df.groupby('Category')['TotalAmount'].sum()   # Sum per category
df.groupby('Category')['TotalAmount'].mean()  # Average per category
df.groupby('Category').size()                 # Count per category
```

### The GroupBy Flow:

```
Original Data:                    Grouped by Category:
┌──────────┬──────────┐            ┌──────────┬───────┐
| Category | Amount   |            | Category | Sum   |
,
100
500
,
200
800
,
300
300
,
100
,
300
,
,
,
,
,
,
,
,
,
,
,
,
,
1
,
,
,
2
,
3

In [ ]:
# ============================================
# MISSION 2: Remove, Rename, Reset
# ============================================

# 1. Drop the Discount column
df_no_disc = df.drop('Discount', axis=1)
print("=== After dropping 'Discount' ===")
print(f"Columns: {df_no_disc.columns.tolist()}")

# 2. Rename TotalAmount to Revenue
df_renamed = df_no_disc.rename(columns={'TotalAmount': 'Revenue'})
print("\n=== After renaming ===")
print(f"Columns: {df_renamed.columns.tolist()}")

# 3. Set CustomerName as index
df_indexed = df_renamed.set_index('CustomerName')
print("\n=== After set_index ===")
print(df_indexed.head())

# 4. Reset index back to default
df_reset = df_indexed.reset_index()
print("\n=== After reset_index ===")
print(df_reset.head())

In [ ]:
# ============================================
# MISSION 3: Change Data Types
# ============================================

# 1. Check current dtypes
print("=== Current Dtypes ===")
print(df.dtypes)

# 2. Convert Quantity to float
df['Quantity'] = df['Quantity'].astype(float)

# 3. Convert Rating to int
df['Rating'] = df['Rating'].astype(int)

# 4. Verify changes
print("\n=== Updated Dtypes ===")
print(df[['Quantity', 'Rating']].dtypes)

In [ ]:
# ============================================
# MISSION 4: GroupBy Basics
# ============================================

# Reset index for easier grouping
df_g = df.reset_index()

# 1. Sum per Category
print("=== Sum of TotalAmount by Category ===")
cat_sum = df_g.groupby('Category')['TotalAmount'].sum()
print(cat_sum)

# 2. Mean per Category
print("\n=== Mean of TotalAmount by Category ===")
cat_mean = df_g.groupby('Category')['TotalAmount'].mean()
print(cat_mean)

# 3. Count per Category
print("\n=== Count of orders by Category ===")
cat_count = df_g.groupby('Category')['Product'].count()
print(cat_count)

# 4. Size (total rows) per Category
print("\n=== Size per Category ===")
cat_size = df_g.groupby('Category').size()
print(cat_size)

# 5. First and last product per Category
print("\n=== First product per Category ===")
cat_first = df_g.groupby('Category').first()[['Product', 'TotalAmount']]
print(cat_first)

print("\n=== Last product per Category ===")
cat_last = df_g.groupby('Category').last()[['Product', 'TotalAmount']]
print(cat_last)

In [ ]:
# ============================================
# MISSION 5: Advanced GroupBy
# ============================================

# 1. .agg() with multiple aggregations
print("=== .agg(): Multiple stats per Category ===")
cat_agg = df_g.groupby('Category').agg({
    'TotalAmount': ['sum', 'mean', 'min', 'max'],
    'Quantity': 'mean',
    'Rating': 'max'
})
print(cat_agg)

# 2. Pivot table
print("\n=== Pivot Table: Avg TotalAmount by Category × PaymentMethod ===")
pivot = df_g.pivot_table(
    values='TotalAmount',
    index='Category',
    columns='PaymentMethod',
    aggfunc='mean',
    fill_value=0
)
print(pivot)

# 3. Filter groups: categories with avg sale > $150
print("\n=== Categories with average sale > $150 ===")
high_avg = (df_g.groupby('Category')['TotalAmount']
            .mean()
            .loc[lambda x: x > 150]
            .reset_index())
print(high_avg)

In [ ]:
# ============================================
# BONUS: Sorting
# ============================================

# Sort by single column (ascending)
print("=== Sorted by TotalAmount (ascending) ===")
sorted_asc = df_g.sort_values('TotalAmount')
print(sorted_asc[['Product', 'TotalAmount']].head())

# Sort by single column (descending)
print("\n=== Sorted by TotalAmount (descending) ===")
sorted_desc = df_g.sort_values('TotalAmount', ascending=False)
print(sorted_desc[['Product', 'TotalAmount']].head())

# Sort by multiple columns
print("\n=== Sorted by Category (asc) then TotalAmount (desc) ===")
multi_sorted = df_g.sort_values(['Category', 'TotalAmount'], ascending=[True, False])
print(multi_sorted[['Category', 'Product', 'TotalAmount']].head(10))